# AYIM 2026 — osu! 圖譜資料流程

憑證集中在同目錄 `.env`（不可提交）；擷取、快取、清理與驗證由 `ayim_pipeline.py` 統一實作。

輸出維持原 dashboard 的難度表與圖集表欄位，採 **UTC 日期，包含結束日當天**。今天僅涵蓋執行時已公布資料。

In [1]:
from pathlib import Path
import logging
import sys

# 支援從 AYIM 或其上一層啟動 Jupyter。
PROJECT = Path.cwd() if (Path.cwd() / "ayim_pipeline.py").exists() else Path.cwd() / "AYIM"
sys.path.insert(0, str(PROJECT.resolve()))
from ayim_pipeline import run
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

In [2]:
START_DATE = "2026-01-01"
END_DATE = "2026-10-23"
REFRESH = False  # False：重用 24 小時內快取；True：全部重新擷取
OUTPUT_DIR = PROJECT / "outputs" / f"{START_DATE}_{END_DATE}"

## 執行

首次執行需取得作者資料與主創 Ranked 歷史，請保留執行至結束。成功回應即寫入快取，中斷後可重跑續接。網路請求遵守官方每分鐘不超過 60 次的建議。

`new_mapper` 表示主創首次擁有 Ranked／Approved **圖集**的年份為該年（合併所有模式）。只有 GD 或 Loved 不算首次 Ranked；刪除或受限帳號可能保留未知值。

In [3]:
result = run(START_DATE, END_DATE, output_dir=OUTPUT_DIR, refresh=REFRESH)
df_api = result["raw"]
df_rank_map_overview = result["difficulties"]
df_rank_mapset_overview = result["beatmapsets"]

2026-10-23 23:40:28,250 Listing page 1: 2026-01-10 17:24:18+00:00, 498 unique difficulties


2026-10-23 23:40:28,265 Listing page 2: 2026-01-18 10:45:59+00:00, 997 unique difficulties


2026-10-23 23:40:28,278 Listing page 3: 2026-01-26 14:46:15+00:00, 1495 unique difficulties


2026-10-23 23:40:28,292 Listing page 4: 2026-02-03 20:24:49+00:00, 1992 unique difficulties


2026-10-23 23:40:28,307 Listing page 5: 2026-02-14 04:23:30+00:00, 2487 unique difficulties


2026-10-23 23:40:28,320 Listing page 6: 2026-02-19 18:23:16+00:00, 2984 unique difficulties


... (204 outputs left out)

2026-10-23 23:40:37,960 Mapper history: 550 / 584 (total hosts 1898)


2026-10-23 23:40:37,978 Mapper history: 575 / 584 (total hosts 1898)


2026-10-23 23:40:37,984 Mapper history: 584 / 584 (total hosts 1898)


2026-10-23 23:40:39,455 Done: 22217 difficulties, 6004 unique sets; output C:\Users\user\python\github\AYIM\outputs\2026-01-01_2026-10-23


In [4]:
manifest = result["manifest"]
{k: manifest[k] for k in ["difficulties", "unique_beatmapsets", "beatmapset_mode_rows", "hosts", "new_hosts", "first_approved_utc", "last_approved_utc", "validation"]}

{'difficulties': 22217,
 'unique_beatmapsets': 6004,
 'beatmapset_mode_rows': 6140,
 'hosts': 1898,
 'new_hosts': 343,
 'first_approved_utc': '2026-01-01 00:42:23',
 'last_approved_utc': '2026-10-23 11:43:47',
 'validation': 'passed'}

In [5]:
df_rank_map_overview.head()

,mode,beatmapset_id,beatmap_id,genre_id,language_id,status,title_unicode,artist_unicode,approved_date,version,difficulty_rating,creator_id,gder_id,gd,favourite_count,playcount,is_new,country,mapper
0,osu!mania,2273158,4842597,Video Game,Japanese,Rank,モエチャッカファイア,Kotoha,2026-01-01 00:42:23,Hyper,Hard,25868133,25868133,0,150,7102,True,CN,Kagetayori
1,osu!mania,2273158,5271245,Video Game,Japanese,Rank,モエチャッカファイア,Kotoha,2026-01-01 00:42:23,Moe,Expert,25868133,25868133,0,150,3808,True,CN,Kagetayori
2,osu!mania,2273158,5359730,Video Game,Japanese,Rank,モエチャッカファイア,Kotoha,2026-01-01 00:42:23,Insane,Insane,25868133,25868133,0,150,4591,True,CN,Kagetayori
3,osu!mania,2273158,5396181,Video Game,Japanese,Rank,モエチャッカファイア,Kotoha,2026-01-01 00:42:23,Hard,Hard,25868133,25868133,0,150,8641,True,CN,Kagetayori
4,osu!mania,2273158,5396273,Video Game,Japanese,Rank,モエチャッカファイア,Kotoha,2026-01-01 00:42:23,Normal,Normal,25868133,25868133,0,150,9784,True,CN,Kagetayori


In [6]:
df_rank_mapset_overview.head()

,mode,beatmapset_id,beatmap_count,title,artist,mapper,mapper_id,approved_date,favourite_count,playcount,genre,language,country,status,new_mapper
0,osu!mania,2273158,5,モエチャッカファイア,Kotoha,Kagetayori,25868133,2026-01-01 00:42:23,150,33926,Video Game,Japanese,CN,Rank,True
1,osu!,2461997,5,心に雲を持つ少年,サニーデイ・サービス,moph,2233878,2026-01-01 00:44:52,347,59198,Rock,Japanese,NZ,Rank,False
2,osu!mania,2441237,3,めうめうぺったんたん！！,芽兎めう (CV: 五十嵐裕美),McEndu,14059135,2026-01-01 01:22:51,87,21360,Video Game,Japanese,CN,Rank,False
3,osu!taiko,2389767,5,nicotine (w/ nomu) -Juztan x Rosh Remix-,blackwinterwells,kageminori,9274708,2026-01-01 04:46:12,22,3715,Electronic,English,RU,Rank,False
4,osu!taiko,2382027,5,DIA DELÍCIA (Sped Up Ver.),NAKAMA & Mc Staff,swake,12967997,2026-01-01 04:46:25,45,13766,Electronic,Other,BR,Rank,False


## 匯入 dashboard

使用 `outputs/<開始日>_<結束日>/` 裡的兩份 CSV：
- `ayim2026_gd_*.csv`：每個難度一列，`mapper` 是難度作者。
- `ayim2026_*.csv`：每個圖集 × 模式一列，`mapper` 是圖集主創。

`playcount`／`favourite_count` 是抓取時累計值。多模式圖集的收藏數會在各模式列重複，跨模式不能直接加總。`manifest.json` 記錄來源時間、列數與檢查結果。詳細修正見 `README.md`。